# HumanAI Detect - On Isleme (Asama 2, Colab GPU)

**Amac:** `human`, `ai_raw`, `ai_humanized` siniflari icin Asama 2 on islemeyi
(Stanza dilbilimsel analiz + BERTurk pseudo-perplexity + burstiness) Colab T4 GPU
uzerinde calistirmak. Yerel CPU'da tek ornek ~200 saniye surdugu icin (maskeli-LM
perplexity yontemi) bu adim GPU gerektiriyor.

**Adimlar:**
1. GPU kontrolu
2. Repo'yu klonla, bagimliliklari kur
3. Ham veriyi dogrudan yerel makineden yukle (`human.jsonl`, `ai_raw.jsonl`, `ai_humanized.jsonl`)
4. `preprocess.py --limit 500` calistir (her sinif icin 500 ornek)
5. Sonuclari dogrudan yerel makineye indir

**Onkosul:** Yukleme hucresi calistirildiginda acilan dosya sec penceresinden
su 3 dosyayi (tek seferde, hepsini birden secerek) yukle:
- `data/raw/human/human.jsonl` (500 kayit)
- `data/raw/ai_raw/ai_raw.jsonl` (3000 kayit, ilk 500'u kullanilacak)
- `data/raw/ai_humanized/ai_humanized.jsonl` (3000 kayit, ilk 500'u kullanilacak)


In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')


In [ ]:
# 3a. Repo klonla (GitHub URL'ini kendi repo'nla degistir)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect


In [ ]:
# 3b. Bagimliliklari kur
!pip install -e '.[dev,colab]' -q
!pip install bitsandbytes accelerate -q


In [ ]:
# 3b-2. Stanza Turkce model indir (pip install'dan sonra AYRI hucrede calistir,
# ayni hucrede yapinca Colab yeni kurulan paketi bazen taniyamiyor)
import stanza
stanza.download('tr')


In [ ]:
# 3c. .env olustur (bu adim icin API key gerekmez)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')


In [ ]:
# 4. Ham veriyi yerel makineden dogrudan yukle
# Acilan pencereden human.jsonl, ai_raw.jsonl, ai_humanized.jsonl dosyalarini
# (hangi isimle olursa olsun, adlarindan taninir) hep birlikte sec.
import os
from pathlib import Path
from google.colab import files

REPO_ROOT = Path('/content/humanai_detect')
os.chdir(REPO_ROOT)  # calisma dizini onceki hucrelerde degismis olsa bile garanti alinir

uploaded = files.upload()

# Sira onemli: 'ai_humanized' iceriginde 'human' gectiği icin once ozel
# etiketler, en sonda genel 'human' kontrol edilir.
for fname in uploaded:
    for label in ['ai_humanized', 'ai_raw', 'human']:
        if label in fname:
            dst = REPO_ROOT / f'data/raw/{label}/{label}.jsonl'
            dst.parent.mkdir(parents=True, exist_ok=True)
            Path(fname).resolve().replace(dst)
            lines = sum(1 for _ in open(dst, encoding='utf-8'))
            print(f'{label}: {lines} kayit -> {dst}')
            break
    else:
        print(f'!!! taninmayan dosya: {fname} (adinda human/ai_raw/ai_humanized gecmiyor)')


In [ ]:
# 5. On isleme (her sinif icin 500 ornek) -- GPU'da calisir, dbmdz/bert-base-turkish-cased otomatik indirilir
%cd /content/humanai_detect
!python scripts/preprocess.py --label all --limit 500


In [ ]:
# 6. Sonuclari dogrudan yerel makineye indir (her dosya icin ayri indirme penceresi acilir)
from pathlib import Path
from google.colab import files

REPO_ROOT = Path('/content/humanai_detect')

for label in ['human', 'ai_raw', 'ai_humanized']:
    src = REPO_ROOT / f'data/interim/{label}/{label}.jsonl'
    if src.exists():
        lines = sum(1 for _ in open(src, encoding='utf-8'))
        print(f'{label}: {lines} ornek -> indiriliyor')
        files.download(str(src))
    else:
        print(f'{label}: dosya bulunamadi! ({src})')


## Yerel Makineye Aktarim

Son hucre calisinca tarayici 3 dosyayi (human.jsonl, ai_raw.jsonl, ai_humanized.jsonl)
tek tek indirir (indirilenler dosyalar genelde Indirilenler/Downloads klasorune duser).
Sonra yerel projede suraya koy (mevcut dosyalarin uzerine yaz):
   - `data/interim/human/human.jsonl`
   - `data/interim/ai_raw/ai_raw.jsonl`
   - `data/interim/ai_humanized/ai_humanized.jsonl`

Ardindan `build_features.py` ve `train_model.py` calistirilarak 500'luk performans
olcumu yapilabilir.
